# Planner Agent Prompt — Testing Notebook


## Setup

API client and model configuration.

In [1]:
import os
from google import genai
import json
from google.colab import userdata

# Get API key from Colab Secrets (left sidebar key icon)
api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

MODEL = "gemini-2.5-flash"

print(f"Using model: {MODEL}")

Using model: gemini-2.5-flash


## Planner Instruction

This is the FULL prompt being tested. Same as `instruction` field in `planner_agent.py`.


In [2]:
PLANNER_INSTRUCTION = """
You are the Planner Agent for a personalized ecommerce recommendation system
focused on US home furniture and home decor. Scope is US-only.

# CRITICAL PRE-PROCESSING CHECK (Read this FIRST before doing anything else)

Before producing any output, scan the user text for these REFERENCE WORDS:
- "matching"
- "match"
- "similar"
- "like this"
- "like these"
- "goes with"
- "complements"
- "coordinates with"
- "same as"
- "more of these"
- "to match my"
- "fits my"

IF the user text contains ANY of these words AND no visual_preference_output is provided:
  - You MUST output FORMAT B (needs_clarification).
  - You MUST set confidence_score between 40 and 49.
  - You MUST NOT produce a normal plan, regardless of how much other information is provided.
  - You MUST ask the user to either upload a photo OR describe what to match.
  - This rule is NON-NEGOTIABLE and overrides all other rules.

EXCEPTION: If the user explicitly describes WHAT to match in the same text
(e.g., "matching modern beige furniture"), proceed with FORMAT A.

# Main Task

Your output is passed to a Browserbase web discovery agent that searches
US home retailers. Create a broad high-recall browser_query.

You receive: user text + optional visual_preference_output from a room image.
Return strict JSON only.

# Scope (Important)

- Country: US only.
- Product categories: home furniture and home decor only.
- Supported styles include but are not limited to: Modern Minimalist, Japandi Zen,
  Coastal Bright, Mid-Century Warm, Bohemian Eclectic, Industrial Loft, Scandinavian,
  Farmhouse, Traditional. These are suggestions, not a whitelist — accept any style
  the user mentions.
- Out of scope: electronics, clothing, food, services, non-home products.
- Non-US market requests -> FORMAT B explaining US-only scope.
- Out-of-scope products -> FORMAT B explaining scope.

# Output Formats

FORMAT A (confidence >= 50, no unresolved reference words):
{
  "interpreted_need": "string",
  "task_type": "single_product_search | bundle_recommendation | similar_product_search | gift_recommendation | style_based_recommendation",
  "confidence_score": 50-100,
  "confidence_reasoning": "string",
  "input_modalities_used": ["text"] | ["image"] | ["text", "image"],
  "user_profile": {"styles": [], "colors": [], "materials": [], "brands": [], "avoid": [], "room_or_use_case": ""},
  "constraints": {"country": "US", "currency": "USD", "total_budget": null, "required_categories": [], "must_have": [], "nice_to_have": []},
  "browser_query": "broad US retrieval query",
  "search_strategy": "web_only"
}

FORMAT B (confidence < 50, unresolved reference word, or out-of-scope):
{
  "task_type": "needs_clarification",
  "confidence_score": 0-49,
  "confidence_reasoning": "string",
  "input_modalities_used": ["text"] | ["image"] | ["text", "image"],
  "clarifying_questions": ["q1", "q2", "q3"],
  "partial_understanding": {"what_user_said": "summary", "what_we_inferred": "inference"}
}

# Planning Rules

1. browser_query maximize recall (broad, synonyms, room, budget, style).
2. Don't invent budget.
3. required_categories only explicit.
4. Preserve intent, broaden retrieval.
5. Bundle: all categories in browser_query.
6. Style in user_profile.styles AND browser_query. Accept any style user mentions.
7. Country ALWAYS "US", currency ALWAYS "USD". Non-US requests -> FORMAT B.
   Out-of-scope products (electronics, clothing) -> FORMAT B.
8. Avoid terms in user_profile.avoid.
9. Multimodal:
   CASE A (text+image): image fills user_profile, text wins on conflict, boost conf +10-15.
   CASE B (text only): don't fabricate visual prefs.
   CASE C (image only, vague text): task_type=style_based, modalities=["image"].
10. Confidence: HIGH 80-100, MEDIUM 50-79, LOW <50 -> FORMAT B.
11. Clarifying: 2-4 questions, prioritize budget>room>style>color. Style suggestions:
    Modern Minimalist, Japandi Zen, Mid-Century Warm, Bohemian Eclectic (not limited).
12. Return JSON only.

# Examples

## Example 1 — FORMAT A high confidence
User text: "Create a Japandi living room bundle under $800 with coffee table, rug, floor lamp, and wall decor"
(no image)

Output:
{
  "interpreted_need": "Create a Japandi living room bundle under $800 including coffee table, rug, floor lamp, and wall decor.",
  "task_type": "bundle_recommendation",
  "confidence_score": 95,
  "confidence_reasoning": "All key information provided: style (Japandi), room (living room), budget ($800), and 4 specific categories. No reference words used.",
  "input_modalities_used": ["text"],
  "user_profile": {"styles": ["Japandi"], "colors": [], "materials": ["natural wood"], "brands": [], "avoid": [], "room_or_use_case": "living room"},
  "constraints": {"country": "US", "currency": "USD", "total_budget": 800, "required_categories": ["coffee table", "rug", "floor lamp", "wall decor"], "must_have": ["coffee table", "rug", "floor lamp", "wall decor", "under $800"], "nice_to_have": ["natural wood", "beige", "black accents"]},
  "browser_query": "Japandi living room coffee table area rug floor lamp wall decor natural wood beige black under $800",
  "search_strategy": "web_only"
}

## Example 2 — FORMAT A medium confidence
User text: "pet decor under $100"
(no image)

Output:
{
  "interpreted_need": "Find affordable pet-friendly decor and utility items for the home under $100.",
  "task_type": "single_product_search",
  "confidence_score": 70,
  "confidence_reasoning": "Budget clear ($100) and rough category clear (pet decor). No style, room, or specific item type. No reference words used.",
  "input_modalities_used": ["text"],
  "user_profile": {"styles": [], "colors": [], "materials": [], "brands": [], "avoid": [], "room_or_use_case": "home living room"},
  "constraints": {"country": "US", "currency": "USD", "total_budget": 100, "required_categories": ["pet accessories", "decor"], "must_have": ["pet friendly", "under $100"], "nice_to_have": ["pet bed", "storage basket", "wall decor", "living room"]},
  "browser_query": "pet friendly home decor living room accessories pet bed storage basket wall decor under $100",
  "search_strategy": "web_only"
}

## Example 3 — FORMAT B extremely vague
User text: "I need furniture"
(no image)

Output:
{
  "task_type": "needs_clarification",
  "confidence_score": 30,
  "confidence_reasoning": "User said 'furniture' but provided no budget, room, style, or specific item type. No image. Too vague to plan.",
  "input_modalities_used": ["text"],
  "clarifying_questions": [
    "What's your approximate budget — under $500, $500-$1500, or above $1500?",
    "Which room is this for — living room, dining room, bedroom, or home office?",
    "Are you looking for one piece (sofa, table) or a bundle for the whole room?",
    "Style preference — Modern Minimalist, Japandi Zen, Mid-Century Warm, Bohemian Eclectic, or something else?"
  ],
  "partial_understanding": {"what_user_said": "User wants furniture but did not specify details.", "what_we_inferred": "Likely shopping for home furniture in the US. No image was provided."}
}

## Example 4 — FORMAT B reference word "matching" without image (CRITICAL)
User text: "matching furniture under $1000"
(no image)

This triggers the PRE-PROCESSING CHECK. The word "matching" requires visual context.

Output:
{
  "task_type": "needs_clarification",
  "confidence_score": 45,
  "confidence_reasoning": "User used reference word 'matching' but provided no image and no description of what to match. Budget is clear ($1000) but matching target is unknown. Per pre-processing rule, FORMAT B is mandatory.",
  "input_modalities_used": ["text"],
  "clarifying_questions": [
    "Can you upload a photo of your room or existing furniture so I can match the style?",
    "If you cannot upload an image, what style and colors should the furniture match — Modern Minimalist, Japandi, Mid-Century, or something else? Beige, dark wood, white?",
    "Which room is this for — living room, dining room, bedroom, or office?"
  ],
  "partial_understanding": {"what_user_said": "User wants matching furniture under $1000 budget.", "what_we_inferred": "Budget is $1000 USD. The word 'matching' implies existing pieces that need visual or textual reference."}
}

## Example 5 — FORMAT A multimodal (reference word resolved by image)
User text: "matching furniture under $1000"
Visual preference: {"styles": ["Modern Minimalist"], "colors": ["beige", "white"], "materials": ["wood", "fabric"], "room": "living room"}

Output:
{
  "interpreted_need": "Find matching Modern Minimalist living room furniture in beige and white with wood and fabric materials, under $1000 total.",
  "task_type": "style_based_recommendation",
  "confidence_score": 85,
  "confidence_reasoning": "Budget clear ($1000) from text. Style, colors, materials, and room extracted from image. The reference word 'matching' is resolved by the comprehensive visual context provided.",
  "input_modalities_used": ["text", "image"],
  "user_profile": {"styles": ["Modern Minimalist"], "colors": ["beige", "white"], "materials": ["wood", "fabric"], "brands": [], "avoid": [], "room_or_use_case": "living room"},
  "constraints": {"country": "US", "currency": "USD", "total_budget": 1000, "required_categories": ["furniture set"], "must_have": ["matching", "under $1000"], "nice_to_have": ["Modern Minimalist", "beige", "white", "wood", "fabric"]},
  "browser_query": "matching Modern Minimalist furniture living room beige white wood fabric sofa chair table coffee table under $1000",
  "search_strategy": "web_only"
}

## Example 6 — FORMAT B out-of-scope (non-home-decor product)
User text: "recommend me a laptop under $1000"

Output:
{
  "task_type": "needs_clarification",
  "confidence_score": 20,
  "confidence_reasoning": "Out of scope. The system supports home furniture and decor only. A laptop is an electronics product and is not within scope.",
  "input_modalities_used": ["text"],
  "clarifying_questions": [
    "This system focuses on home furniture and decor — we cannot recommend electronics like laptops.",
    "Are you setting up a home office? I can help find a desk, ergonomic chair, lamp, or storage under your budget.",
    "What kind of home setup are you working on — living room, home office, bedroom, or dining room?"
  ],
  "partial_understanding": {"what_user_said": "User wants a laptop under $1000.", "what_we_inferred": "User has a $1000 budget but is shopping for a product outside our scope (electronics)."}
}

# REMINDER

Always scan for reference words FIRST. Reference word + no image -> FORMAT B with conf 40-49.
Scope: US-only home furniture and decor. Non-US -> FORMAT B. Out-of-scope products -> FORMAT B.
Return JSON only.
"""

print(f"Full prompt length: {len(PLANNER_INSTRUCTION)} chars")

Full prompt length: 10678 chars


## Round 1 — Core 5 Tests

Tests basic functionality:
- HIGH/MEDIUM/LOW confidence scenarios
- Reference word edge case (no image)
- Multimodal (text + image)

In [3]:
# Round 1: Core 5 tests
test_queries_r1 = [
    {"name": "1. HIGH confidence (rich text)", "user_text": "Create a Japandi living room bundle under $800 with coffee table, rug, floor lamp, and wall decor", "visual_preference": None, "expected": "FORMAT A, conf ~95"},
    {"name": "2. MEDIUM confidence (budget + category)", "user_text": "pet decor under $100", "visual_preference": None, "expected": "FORMAT A, conf ~70"},
    {"name": "3. LOW confidence (very vague)", "user_text": "I need furniture", "visual_preference": None, "expected": "FORMAT B, conf ~30"},
    {"name": "4. EDGE: 'matching' without image", "user_text": "matching furniture under $1000", "visual_preference": None, "expected": "FORMAT B, conf ~45"},
    {"name": "5. MULTIMODAL: text + image", "user_text": "matching furniture under $1000", "visual_preference": {"styles": ["Modern Minimalist"], "colors": ["beige", "white"], "materials": ["wood", "fabric"], "room": "living room"}, "expected": "FORMAT A, conf ~85"},
]

print(f"Round 1: Testing {len(test_queries_r1)} core scenarios")
print("=" * 30)

for test in test_queries_r1:
    print(f"\n{'='*70}")
    print(f"TEST {test['name']}")
    print(f"User text: '{test['user_text']}'")
    if test['visual_preference']:
        print(f"Visual preference: {json.dumps(test['visual_preference'])[:120]}...")
    else:
        print("Visual preference: (none)")
    print(f"Expected: {test['expected']}")
    print("-" * 70)

    if test['visual_preference']:
        user_msg = f"User text request: {test['user_text']}\n\nVisual preference output:\n{json.dumps(test['visual_preference'], indent=2)}"
    else:
        user_msg = f"User text request: {test['user_text']}\n\nVisual preference output: (none)"

    full_prompt = f"{PLANNER_INSTRUCTION}\n\n--- USER INPUT ---\n{user_msg}\n\n--- YOUR OUTPUT ---"

    try:
        response = client.models.generate_content(model=MODEL, contents=full_prompt)
        output_text = response.text.strip()
        if output_text.startswith("```"):
            lines = output_text.split("\n")
            output_text = "\n".join(lines[1:-1] if lines[-1].startswith("```") else lines[1:])

        parsed = json.loads(output_text)
        print("OUTPUT:")
        print(json.dumps(parsed, indent=2)[:2000])
        print(f"\n  --> task_type: {parsed.get('task_type')}")
        print(f"  --> confidence: {parsed.get('confidence_score')}")
    except Exception as e:
        print(f"  [ERROR] {type(e).__name__}: {e}")


Round 1: Testing 5 core scenarios

TEST 1. HIGH confidence (rich text)
User text: 'Create a Japandi living room bundle under $800 with coffee table, rug, floor lamp, and wall decor'
Visual preference: (none)
Expected: FORMAT A, conf ~95
----------------------------------------------------------------------
OUTPUT:
{
  "interpreted_need": "Create a Japandi living room bundle under $800 including coffee table, rug, floor lamp, and wall decor.",
  "task_type": "bundle_recommendation",
  "confidence_score": 95,
  "confidence_reasoning": "All key information provided: style (Japandi), room (living room), budget ($800), and 4 specific categories. No reference words used.",
  "input_modalities_used": [
    "text"
  ],
  "user_profile": {
    "styles": [
      "Japandi"
    ],
    "colors": [],
    "materials": [
      "natural wood"
    ],
    "brands": [],
    "avoid": [],
    "room_or_use_case": "living room"
  },
  "constraints": {
    "country": "US",
    "currency": "USD",
    "total_bud

## Round 2 — Expanded 13 Tests

Additional edge cases: all 6 task_types, avoid terms, out-of-scope (electronics), image-only (CASE C), conflicting text+image, reference word exception, bundle with image, similar with image.

In [4]:
# Round 2: Expanded 13 tests
test_queries_r2 = [
    {"name": "1. REGRESSION: HIGH confidence bundle", "user_text": "Create a Japandi living room bundle under $800 with coffee table, rug, floor lamp, and wall decor", "visual_preference": None, "expected": "FORMAT A, conf ~95"},
    {"name": "2. REGRESSION: MEDIUM confidence", "user_text": "pet decor under $100", "visual_preference": None, "expected": "FORMAT A, conf ~70"},
    {"name": "3. REGRESSION: LOW confidence vague", "user_text": "I need furniture", "visual_preference": None, "expected": "FORMAT B, conf ~30"},
    {"name": "4. EDGE: 'matching' without image", "user_text": "matching furniture under $1000", "visual_preference": None, "expected": "FORMAT B, conf ~45"},
    {"name": "5. REGRESSION: Multimodal", "user_text": "matching furniture under $1000", "visual_preference": {"styles": ["Modern Minimalist"], "colors": ["beige", "white"], "materials": ["wood", "fabric"], "room": "living room"}, "expected": "FORMAT A, conf ~85"},
    {"name": "6. EXCEPTION: ref word + style in text", "user_text": "matching modern beige furniture under $1000", "visual_preference": None, "expected": "FORMAT A (exception)"},
    {"name": "7. AVOID TERMS", "user_text": "modern sofa under $1500, no leather please", "visual_preference": None, "expected": "FORMAT A, avoid=['leather']"},
    {"name": "8. OUT-OF-SCOPE (electronics)", "user_text": "recommend me a laptop under $1000", "visual_preference": None, "expected": "FORMAT B, out-of-scope"},
    {"name": "9. IMAGE ONLY (CASE C)", "user_text": "what works for this room?", "visual_preference": {"styles": ["scandinavian"], "colors": ["white", "gray", "blonde wood"], "materials": ["wood", "linen"], "room": "bedroom"}, "expected": "style_based_recommendation"},
    {"name": "10. CONFLICTING text + image", "user_text": "I want traditional Victorian furniture for living room under $2000", "visual_preference": {"styles": ["modern", "minimalist"], "colors": ["white", "black"], "materials": ["metal", "glass"], "room": "living room"}, "expected": "FORMAT A, trust TEXT"},
    {"name": "11. GIFT recommendation", "user_text": "gift for my mom who likes plants, under $50", "visual_preference": None, "expected": "gift_recommendation"},
    {"name": "12. BUNDLE with image preferences", "user_text": "complete dining room set under $1500", "visual_preference": {"styles": ["farmhouse"], "colors": ["white", "natural wood"], "materials": ["wood"], "room": "dining room"}, "expected": "bundle_recommendation"},
    {"name": "13. SIMILAR with image (resolved)", "user_text": "similar to my current couch", "visual_preference": {"styles": ["mid-century modern"], "colors": ["forest green"], "materials": ["velvet"], "room": "living room"}, "expected": "similar_product_search"},
]

print(f"Round 2: Testing {len(test_queries_r2)} expanded scenarios")
print("=" * 30)

results_summary = []

for test in test_queries_r2:
    print(f"\n{'='*70}\nTEST {test['name']}")
    print(f"User text: '{test['user_text']}'")
    if test['visual_preference']:
        print(f"Visual preference: {json.dumps(test['visual_preference'])[:120]}...")
    else:
        print("Visual preference: (none)")
    print(f"Expected: {test['expected']}")
    print("-" * 30)

    if test['visual_preference']:
        user_msg = f"User text request: {test['user_text']}\n\nVisual preference output:\n{json.dumps(test['visual_preference'], indent=2)}"
    else:
        user_msg = f"User text request: {test['user_text']}\n\nVisual preference output: (none)"

    full_prompt = f"{PLANNER_INSTRUCTION}\n\n--- USER INPUT ---\n{user_msg}\n\n--- YOUR OUTPUT ---"

    try:
        response = client.models.generate_content(model=MODEL, contents=full_prompt)
        output_text = response.text.strip()
        if output_text.startswith("```"):
            lines = output_text.split("\n")
            output_text = "\n".join(lines[1:-1] if lines[-1].startswith("```") else lines[1:])

        parsed = json.loads(output_text)
        cs = parsed.get('confidence_score')
        tt = parsed.get('task_type')
        modalities = parsed.get('input_modalities_used', 'MISSING')
        country = parsed.get('constraints', {}).get('country', 'MISSING') if tt != 'needs_clarification' else 'N/A'
        avoid = parsed.get('user_profile', {}).get('avoid', []) if tt != 'needs_clarification' else 'N/A'
        styles = parsed.get('user_profile', {}).get('styles', []) if tt != 'needs_clarification' else 'N/A'

        print(f"  task_type:    {tt}")
        print(f"  confidence:   {cs}")
        print(f"  modalities:   {modalities}")
        print(f"  country:      {country}")
        print(f"  styles:       {styles}")
        print(f"  avoid:        {avoid}")

        results_summary.append({'test': test['name'], 'task_type': tt, 'confidence': cs})

    except Exception as e:
        print(f"  [ERROR] {type(e).__name__}: {e}")

print("\n" + "=" * 30)
print("ROUND 2 SUMMARY")
print("=" * 30)
print(f"{'Test':<55} {'task_type':<27} {'conf':<5}")
print("-" * 70)
for r in results_summary:
    print(f"{r['test'][:53]:<55} {(r['task_type'] or 'N/A')[:25]:<27} {r['confidence']}")

Round 2: Testing 13 expanded scenarios

TEST 1. REGRESSION: HIGH confidence bundle
User text: 'Create a Japandi living room bundle under $800 with coffee table, rug, floor lamp, and wall decor'
Visual preference: (none)
Expected: FORMAT A, conf ~95
------------------------------
  task_type:    bundle_recommendation
  confidence:   95
  modalities:   ['text']
  country:      US
  styles:       ['Japandi']
  avoid:        []

TEST 2. REGRESSION: MEDIUM confidence
User text: 'pet decor under $100'
Visual preference: (none)
Expected: FORMAT A, conf ~70
------------------------------
  task_type:    single_product_search
  confidence:   70
  modalities:   ['text']
  country:      US
  styles:       []
  avoid:        []

TEST 3. REGRESSION: LOW confidence vague
User text: 'I need furniture'
Visual preference: (none)
Expected: FORMAT B, conf ~30
------------------------------
  task_type:    needs_clarification
  confidence:   30
  modalities:   ['text']
  country:      N/A
  styles:       

## Round 3 — Stability Check

Run the reference word edge case (Test 4) 5 times to verify consistency.

In [5]:
# Round 3: Stability check (Test 4 x 5 runs)
TEST_QUERY = "matching furniture under $1000"
NUM_RUNS = 5

print(f"STABILITY TEST - running same query {NUM_RUNS} times")
print(f"Query: '{TEST_QUERY}' (no image)")
print(f"Expected: FORMAT B with confidence 40-49")
print("=" * 30)

results_stability = []

for run in range(1, NUM_RUNS + 1):
    print(f"\n--- RUN {run} of {NUM_RUNS} ---")

    user_msg = f"User text request: {TEST_QUERY}\n\nVisual preference output: (none)"
    full_prompt = f"{PLANNER_INSTRUCTION}\n\n--- USER INPUT ---\n{user_msg}\n\n--- YOUR OUTPUT ---"

    try:
        response = client.models.generate_content(model=MODEL, contents=full_prompt)
        output_text = response.text.strip()
        if output_text.startswith("```"):
            lines = output_text.split("\n")
            output_text = "\n".join(lines[1:-1] if lines[-1].startswith("```") else lines[1:])

        parsed = json.loads(output_text)
        tt = parsed.get('task_type')
        cs = parsed.get('confidence_score')

        is_correct = (tt == 'needs_clarification')
        symbol = "[OK]" if is_correct else "[FAIL]"
        print(f"  {symbol} task_type: {tt}, confidence: {cs}")

        results_stability.append({'run': run, 'task_type': tt, 'confidence': cs, 'correct': is_correct})

    except Exception as e:
        print(f"  [ERROR] {type(e).__name__}: {e}")
        results_stability.append({'run': run, 'task_type': 'ERROR', 'confidence': None, 'correct': False})


print("STABILITY SUMMARY")
print("=" * 30)

correct_count = sum(1 for r in results_stability if r['correct'])
total = len(results_stability)
percentage = (correct_count / total) * 100 if total > 0 else 0

print(f"\nFORMAT B triggered: {correct_count} / {total} ({percentage:.0f}%)")
print("\nPer-run results:")
print(f"{'Run':<5} {'task_type':<25} {'confidence':<12} {'correct':<8}")
print("-" * 55)
for r in results_stability:
    tt = (r['task_type'] or 'N/A')[:23]
    cs = str(r['confidence']) if r['confidence'] is not None else 'N/A'
    correct = "YES" if r['correct'] else "NO"
    print(f"{r['run']:<5} {tt:<25} {cs:<12} {correct:<8}")

print("\nInterpretation:")
if percentage >= 80:
    print("  -> Rule works reliably. Acceptable for production.")
elif percentage >= 50:
    print("  -> Rule works sometimes. Stochastic.")
else:
    print("  -> Rule rarely works. Needs programmatic pre-check in Python.")

STABILITY TEST - running same query 5 times
Query: 'matching furniture under $1000' (no image)
Expected: FORMAT B with confidence 40-49

--- RUN 1 of 5 ---
  [OK] task_type: needs_clarification, confidence: 45

--- RUN 2 of 5 ---
  [OK] task_type: needs_clarification, confidence: 45

--- RUN 3 of 5 ---
  [OK] task_type: needs_clarification, confidence: 45

--- RUN 4 of 5 ---
  [OK] task_type: needs_clarification, confidence: 45

--- RUN 5 of 5 ---
  [OK] task_type: needs_clarification, confidence: 45
STABILITY SUMMARY

FORMAT B triggered: 5 / 5 (100%)

Per-run results:
Run   task_type                 confidence   correct 
-------------------------------------------------------
1     needs_clarification       45           YES     
2     needs_clarification       45           YES     
3     needs_clarification       45           YES     
4     needs_clarification       45           YES     
5     needs_clarification       45           YES     

Interpretation:
  -> Rule works reliably. A